# run the analysis below

In [6]:
import argparse, os, re, sys, time
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

In [7]:
FAMILIES = ["LINE/L1", "LINE/L2", "LTR", "LTR/ERV1", "LTR/ERVK", "LTR/ERVL",
            "LTR/ERVL-MaLR", "Retroposon/SVA", "SINE/Alu", "SINE/MIR"]
ATOI = {"AG", "TC"}          

In [8]:
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

## run part one : 

In [10]:
def aggregate_sprint(sprint_csv, out_edit, out_map, chunksize=2_000_000):
    """ here do analysis over sprint_repeat_overlaps.csv """
    cols = ["run_id", "type", "supporting_reads", "ad", "dp", "rep_name", "class_family"]
    dtypes = {"run_id": "category", "type": "category", "supporting_reads": "int32", "ad": "int32",
              "dp": "int32", "rep_name": "category", "class_family": "category"}
    agg_parts, map_parts, n = [], [], 0
    for chunk in pd.read_csv(sprint_csv, usecols=cols, dtype=dtypes, chunksize=chunksize,
                             keep_default_na=False, na_values=[]):
        n += len(chunk)
        
        # rows with no overlapping repeats have the NA ...... 
        rep = chunk[chunk["class_family"] != "NA"]
        map_parts.append(rep[["rep_name", "class_family"]].astype(str).drop_duplicates())
        ai = rep[rep["type"].isin(ATOI)].copy()
        ai["sr2"] = ai["supporting_reads"] >= 2
        g = ai.groupby(["run_id", "class_family"], observed=True)
        part = pd.DataFrame({"n_sites": g.size(), "sum_ad": g["ad"].sum(), "sum_dp": g["dp"].sum(),
                             "n_sites_sr2": g["sr2"].sum(),
                             "sum_ad_sr2": ai[ai["sr2"]].groupby(["run_id", "class_family"], observed=True)["ad"].sum(),
                             "sum_dp_sr2": ai[ai["sr2"]].groupby(["run_id", "class_family"], observed=True)["dp"].sum()})
        agg_parts.append(part.fillna(0))
        log(f"  SPRINT rows read: {n:,}")
        
    agg = pd.concat(agg_parts).reset_index()
    agg["run_id"] = agg["run_id"].astype(str); agg["class_family"] = agg["class_family"].astype(str)
    agg = agg.groupby(["run_id", "class_family"]).sum().astype("int64").reset_index()
    agg = agg[agg["n_sites"] > 0].sort_values(["run_id", "class_family"])
    agg.to_csv(out_edit, index=False)
    m = pd.concat(map_parts).drop_duplicates().sort_values("rep_name")
    dup = m[m.duplicated("rep_name", keep=False)]
    
    if len(dup):
        log(f"  WARNING: {dup['rep_name'].nunique()} repeat names map to >1 family; keeping the first ")
        m = m.drop_duplicates("rep_name")
    m.to_csv(out_map, index=False)
    log(f"  wrote {out_edit} ({len(agg)} rows) and {out_map} ({len(m)} names)")

### run some tests

## run ParT two here below :

In [12]:
def aggregate_counts(counts_csv, map_csv, out_expr, chunksize=1_000_000):
    """to summ the featurecounts fractional counts per class/family; GeneID '<name>_<index>' -> name -> family."""
    name2fam = pd.read_csv(map_csv).set_index("rep_name")["class_family"]
    total, nloci, n = None, None, 0
    
    for chunk in pd.read_csv(counts_csv, chunksize=chunksize):
        n += len(chunk)
        names = chunk["GeneID"].str.replace(r"_\d+$", "", regex=True)
        fam = names.map(name2fam).fillna("UNMAPPED")
        vals = chunk.drop(columns="GeneID").astype("float64")
        part = vals.groupby(fam.values).sum()
        cnt = fam.value_counts()
        total = part if total is None else total.add(part, fill_value=0)
        nloci = cnt if nloci is None else nloci.add(cnt, fill_value=0)
        log(f"  count rows read: {n:,}")
        
    total.columns = [c.replace(".sorted.bam", "") for c in total.columns]
    total.insert(0, "n_loci", nloci.reindex(total.index).astype(int))
    total.index.name = "class_family"
    total.round(3).to_csv(out_expr)
    unm = total.loc["UNMAPPED", "n_loci"] if "UNMAPPED" in total.index else 0
    
    log(f"  wrote {out_expr}; {unm:,} loci could not be assigned a family (names absent from SPRINT file)   ")

In [13]:
def mor_size_factors(M):
    
    """DESeq2::estimateSizeFactorsForMatrix(type = 'ratio'): genes with any zero are excluded."""
    
    with np.errstate(divide="ignore"):
        logM = np.log(M)
    loggeo = logM.mean(axis=1)
    ok = np.isfinite(loggeo)
    
    return np.array([np.exp(np.median((logM[ok, j] - loggeo[ok])[M[ok, j] > 0])) for j in range(M.shape[1])])

In [14]:
def size_factors(salmon_counts_csv, salmon_lengths_csv, stat_csv, sample_sheet, out_csv):
    
    counts = pd.read_csv(salmon_counts_csv, index_col=0)
    lengths = pd.read_csv(salmon_lengths_csv, index_col=0).loc[counts.index, counts.columns]
    C = np.round(counts.values)                                     # DESeqDataSetFromTximport round the counts
    L = lengths.values
    # run the below
    nm = L / np.exp(np.log(L).mean(axis=1, keepdims=True))          # avgTxLength / row geometric mean
    sf_len = mor_size_factors(C / nm)
    nf = nm * sf_len
    nf = nf / np.exp(np.log(nf).mean(axis=1, keepdims=True))
    sf_user = np.exp(np.log(nf).mean(axis=0))
    sf_plain = mor_size_factors(C)
    st = pd.read_csv(stat_csv).set_index("Status")
    st.columns = [c.replace(".sorted.bam", "") for c in st.columns]
    lib = (st.loc["Assigned"] + st.loc["Unassigned_NoFeatures"]).reindex(counts.columns)
    tab = pd.DataFrame({"sf_user_geomeanNF": sf_user, "sf_plain_MoR": sf_plain, "sf_lengthcorr_MoR": sf_len,
                        "sf_total_reads": (lib / np.exp(np.log(lib).mean())).values,
                        "sf_total_genic_counts": C.sum(0) / np.exp(np.log(C.sum(0)).mean())}, index=counts.columns)
    
    ss = sample_sheet.set_index("srr")
    tab.insert(0, "condition", ss.reindex(tab.index)["condition"].values)
    tab.insert(0, "subject", ss.reindex(tab.index)["subject"].values)
    tab.index.name = "srr"
    tab.to_csv(out_csv, float_format="%.6g")
    log(f"  wrote {out_csv}; max |notebook SF / plain MoR - 1| = {np.abs(sf_user / sf_plain - 1).max():.3f}")
    
    return tab

###  run the Benjamini - Hochberg

In [15]:
def bh(p):
    
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    ranked = p[o] * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(adj, 1); return out

In [16]:
def paired_stats(out_dir, sample_sheet, sf_tab):
    
    ss = sample_sheet
    subjects = sorted(ss["subject"].unique())
    srr_of = {(r.subject, r.condition): r.srr for r in ss.itertuples()}
    samples = list(ss["srr"])
    sf_gene = sf_tab["sf_user_geomeanNF"].reindex(samples)
    sf_lib = sf_tab["sf_total_reads"].reindex(samples)
 
    expr = pd.read_csv(f"{out_dir}/expression_by_family.csv").set_index("class_family")
    expr.columns = [c.replace(".sorted.bam", "") for c in expr.columns]
    fam = expr.loc[FAMILIES, samples].astype(float)                 # raw fractional family counts
    ed = pd.read_csv(f"{out_dir}/editing_AI_by_family.csv")
    ed = ed[ed["run_id"].isin(samples)]
    ad = ed.pivot(index="class_family", columns="run_id", values="sum_ad").loc[FAMILIES, samples].astype(float)
    dp = ed.pivot(index="class_family", columns="run_id", values="sum_dp").loc[FAMILIES, samples].astype(float)
 
    metrics = {"expr_geneSF": fam / sf_gene,            # expression, gene - derived size factors 
               "expr_totalreads": fam / sf_lib,         # expression, total-reads scaling (sensitivity)
               "editA_sitelevel": ad / dp,              # editing level at the .... called sites
               "editB_perread": 1000 * ad / fam}        # edited reads per 1000 family reads 
    summary, per_subject = {}, {}
    
    for name, M in metrics.items():
        L = pd.DataFrame({s: np.log2(M[srr_of[(s, "TN")]] / M[srr_of[(s, "TP")]]) for s in subjects})
        per_subject[name] = L
        summary[f"{name}_log2FC"] = L.mean(axis=1)
        summary[f"{name}_sem"] = L.std(axis=1, ddof=1) / np.sqrt(L.shape[1])
        summary[f"{name}_p"] = stats.ttest_1samp(L, 0, axis=1).pvalue
    res = pd.DataFrame(summary, index=FAMILIES)
    for tag, name in (("A", "editA_sitelevel"), ("B", "editB_perread")):
        res[f"{tag}_edit_padj_BH"] = bh(res[f"{name}_p"])
        res[f"{tag}_size_level"] = np.where(res[f"{tag}_edit_padj_BH"] < 0.01, "<0.01",
                                            np.where(res[f"{tag}_edit_padj_BH"] < 0.05, "<0.05", ">=0.05"))
        
    res["mean_EI_sitelevel_pct"] = (100 * ad / dp)[samples].mean(axis=1)
    res["mean_edited_per_1000_reads"] = (1000 * ad / fam)[samples].mean(axis=1)
    res["n_subjects"] = len(subjects)
    res.index.name = "class_family"
    res.to_csv(f"{out_dir}/editing_vs_expression_results.csv", float_format="%.5g")
    pd.concat(per_subject, axis=1).to_csv(f"{out_dir}/editing_vs_expression_per_subject.csv", float_format="%.5g")
    log(f"  wrote {out_dir}/editing_vs_expression_results.csv")
    
    return res

### tes for below 

In [19]:
1

1

In [21]:
## wait examples

In [22]:
SIZE_PT = {">=0.05": 7.5, "<0.05": 11.5, "<0.01": 16.5}          # here we bubble diameters (points) by adjusted p
LEVELS = [">=0.05", "<0.05", "<0.01"]
CMAP = LinearSegmentedColormap.from_list("bwr_gg", ["#2166AC", "#FFFFFF", "#B2182B"])
NORM = Normalize(vmin=-0.7, vmax=0.7)
GRID, INK2, EDGE, SEG = "#EBEBEB", "#4D4D4D", "#4D4D4D", "#7F7F7F"
 
# for the labels below
PANELS = {
    "A": dict(xcol="editA_sitelevel_log2FC", lvl="A_size_level", stem="editing_vs_expression_bubble_A_sitelevel",
              subtitle="Editing rate = editing level at SPRINT AG/TC sites (Σedited ÷ Σtotal reads)\n"
                       "Expression normalised with gene-derived size factors · n = 7 paired subjects",
              xlim=(-0.7, 0.1), ylim=(-0.02, 0.25),
              labels={"LTR": ((-0.460, 0.189), "center"), "Retroposon/SVA": ((-0.345, 0.158), "right"),
                      "LTR/ERVK": ((-0.293, 0.078), "center"), "LTR/ERV1": ((-0.190, 0.114), "right"),
                      "LINE/L1": ((-0.082, 0.136), "left"), "SINE/Alu": ((-0.083, 0.164), "left"),
                      "LTR/ERVL-MaLR": ((-0.230, 0.205), "right"), "LINE/L2": ((-0.195, 0.205), "left"),
                      "LTR/ERVL": ((-0.210, 0.152), "left"), "SINE/MIR": ((-0.235, 0.141), "right")}),
    "B": dict(xcol="editB_perread_log2FC", lvl="B_size_level", stem="editing_vs_expression_bubble_B_perread",
              subtitle="Editing rate = edited reads per family read (notebook editing_index: Σedited ÷ family read count)\n"
                       "Expression normalised with gene-derived size factors · n = 7 paired subjects",
              xlim=(-0.8, 0.15), ylim=(-0.02, 0.25),
              labels={"LTR": ((-0.690, 0.160), "center"), "LTR/ERVK": ((-0.525, 0.076), "center"),
                      "LTR/ERVL-MaLR": ((-0.285, 0.205), "right"), "LTR/ERVL": ((-0.262, 0.160), "right"),
                      "LINE/L1": ((-0.200, 0.118), "right"), "LTR/ERV1": ((-0.105, 0.112), "left"),
                      "SINE/Alu": ((-0.145, 0.146), "left"), "SINE/MIR": ((-0.140, 0.196), "right"),
                      "LINE/L2": ((-0.060, 0.205), "left"), "Retroposon/SVA": ((-0.050, 0.158), "left")}),
}

In [23]:
def bubble_plot(res, tag, out_dir):
    
    P = PANELS[tag]
    x, y, lvl = res[P["xcol"]], res["expr_geneSF_log2FC"], res[P["lvl"]]
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
    fig = plt.figure(figsize=(8.6, 6), dpi=150, facecolor="white")
    ax = fig.add_axes([0.11, 0.12, 0.60, 0.72]); ax.set_facecolor("white")
    ax.grid(True, color=GRID, lw=0.7, zorder=0); ax.set_axisbelow(True)
    for s in ax.spines.values():
        s.set_visible(False)
        
    ax.tick_params(length=0, colors=INK2, labelsize=11)
    ax.axhline(0, color="#999999", lw=0.9, ls=(0, (5, 4)), alpha=0.7, zorder=1)
    ax.axvline(0, color="#999999", lw=0.9, ls=(0, (5, 4)), alpha=0.7, zorder=1)
    for f in x.abs().sort_values(ascending=False).index:          # small bubbles sit on top for seeing better
        d = SIZE_PT[lvl[f]]
        ax.scatter(x[f], y[f], s=d ** 2, color=CMAP(NORM(x[f])), edgecolor=EDGE, linewidth=0.8, alpha=0.85, zorder=3)
        
    for f in x.index:
        (lx, ly), ha = P["labels"].get(f, ((x[f] + 0.02, y[f]), "left"))
        ax.annotate(f, (x[f], y[f]), xytext=(lx, ly), textcoords="data", ha=ha, va="center", fontsize=9.5,
                    color="black", zorder=4,
                    arrowprops=dict(arrowstyle="-", color=SEG, lw=0.6, shrinkA=SIZE_PT[lvl[f]] / 2 + 1, shrinkB=2))
        
    ax.set_xlim(*P["xlim"]); ax.set_ylim(*P["ylim"])
    ax.set_xlabel("Editing-rate (log2 fold-change)", fontsize=12.5, color="black", labelpad=8)
    ax.set_ylabel("Total-count (log2 fold-change)", fontsize=12.5, color="black", labelpad=8)
    ax.set_title("Editing Rate vs. Total Count Fold Change", fontsize=15.5, fontweight="bold", color="black", pad=34)
    ax.text(0.5, 1.02, P["subtitle"], transform=ax.transAxes, ha="center", va="bottom", fontsize=7.6,
            color="#555555", linespacing=1.4)
    ax.text(0.01, 0.985, tag, transform=fig.transFigure, fontsize=15, fontweight="bold", va="top")
    # legends, ggplot-style on the right: colour bar with its title above is there
    cax = fig.add_axes([0.80, 0.50, 0.025, 0.22])
    cb = matplotlib.colorbar.ColorbarBase(cax, cmap=CMAP, norm=NORM, orientation="vertical", ticks=[-0.4, 0, 0.4])
    cb.outline.set_visible(False); cb.ax.tick_params(length=0, labelsize=10.5, colors=INK2)
    cb.ax.set_yticklabels(["-0.4", "0.0", "0.4"])
    fig.text(0.79, 0.745, "Editing-rate log2FC", fontsize=12.5, color="black", va="bottom")
    present = [l for l in LEVELS if (lvl == l).any()]
    handles = [Line2D([], [], marker="o", ls="none", markersize=SIZE_PT[l] * 0.85, markerfacecolor="white",
                      markeredgecolor=EDGE, markeredgewidth=0.8, label=l) for l in present]
    leg = fig.legend(handles=handles, title="Adjusted p-value", loc="upper left", bbox_to_anchor=(0.775, 0.44),
                     frameon=False, fontsize=10.5, title_fontsize=12.5, labelspacing=1.2, handletextpad=0.9,
                     borderaxespad=0)
    
    leg._legend_box.align = "left"
    for ext in ("png", "pdf"):
        fig.savefig(f"{out_dir}/{P['stem']}.{ext}", dpi=200, facecolor="white")
    plt.close(fig)
    
    log(f"  wrote {out_dir}/{P['stem']}.png / .pdf")

## call all the above functions in seq

In [27]:
def run_pipeline(folder, salmon_counts, salmon_lengths, sprint=None, counts=None, stat=None,
                 sample_sheet=None, out="editing_vs_expression", force=False):
    F = folder.rstrip("/")
    out = out if os.path.isabs(out) else f"{F}/{out}"
    os.makedirs(out, exist_ok=True)
    sprint = sprint or f"{F}/sprint_repeat_overlaps.csv"
    counts = counts or f"{F}/liu_repeat_counts-counts.csv"
    
    stat   = stat   or f"{F}/liu_repeat_counts-stat.csv"
    sheet  = pd.read_csv(sample_sheet or f"{F}/liu_MI_sample_sheet.csv")
    
    assert sheet.groupby("subject")["condition"].apply(lambda s: set(s) == {"TP", "TN"}).all(), \
        "every subject needs exactly one TP and one TN sample"

    f_edit, f_map, f_expr, f_sf = (f"{out}/editing_AI_by_family.csv", f"{out}/repname_to_family.csv",
                                   f"{out}/expression_by_family.csv", f"{out}/size_factors_comparison.csv")
    log("step 1: SPRINT editing sites -> per sample x family")
    
    if force or not (os.path.exists(f_edit) and os.path.exists(f_map)):
        aggregate_sprint(sprint, f_edit, f_map)
    else:
        log("  cached")
    log("step 2: featureCounts loci -> per family x sample")
    if force or not os.path.exists(f_expr):
        aggregate_counts(counts, f_map, f_expr)
    else:
        log("  cached")
    log("step 3: size factors from Salmon gene counts")
    sf_tab = size_factors(salmon_counts, salmon_lengths, stat, sheet, f_sf)
    log("step 4: paired statistics")
    res = paired_stats(out, sheet, sf_tab)
    pd.set_option("display.width", 220)
    print(res[["expr_geneSF_log2FC", "expr_geneSF_p", "editA_sitelevel_log2FC", "A_edit_padj_BH", "A_size_level",
               "editB_perread_log2FC", "B_edit_padj_BH", "B_size_level"]].round(4).to_string())
    log("step 5: bubble plots")
    for tag in ("A", "B"):
        bubble_plot(res, tag, out)
    log("done")
    
    return res

In [28]:
res = run_pipeline(
    folder         = "/n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA",   
    salmon_counts  = "/n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/liu_salmon_counts.csv",
    salmon_lengths = "/n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/liu_salmon_lengths.csv",
    sprint="/n/scratch/users/r/rob6090/projects/tdp-43/scripts/STAR_data/sprint_repeat_overlaps.csv", 
)

from IPython.display import Image, display         
for tag in ("A", "B"):
    display(Image(f"{res.attrs.get('out', 'editing_vs_expression')}/{PANELS[tag]['stem']}.png")) if False else None

[10:20:58] step 1: SPRINT editing sites -> per sample x family
[10:21:00]   SPRINT rows read: 2,000,000
[10:21:02]   SPRINT rows read: 4,000,000
[10:21:04]   SPRINT rows read: 6,000,000
[10:21:06]   SPRINT rows read: 8,000,000
[10:21:08]   SPRINT rows read: 10,000,000
[10:21:10]   SPRINT rows read: 12,000,000
[10:21:12]   SPRINT rows read: 14,000,000
[10:21:13]   SPRINT rows read: 16,000,000
[10:21:15]   SPRINT rows read: 18,000,000
[10:21:17]   SPRINT rows read: 20,000,000
[10:21:19]   SPRINT rows read: 22,000,000
[10:21:19]   SPRINT rows read: 22,057,717
[10:21:19]   wrote /n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/editing_vs_expression/editing_AI_by_family.csv (846 rows) and /n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/editing_vs_expression/repname_to_family.csv (1412 names)
[10:21:19] step 2: featureCounts loci -> per family x sample
[10:21:20]   count rows read: 1,000,000
[10:21:22]   count rows read: 2,000,000
[10:21:23]   c